# 03 `pace_select` + Export POSCARs for DFT

This notebook streamlines the post-LAMMPS active-learning workflow for all folders in `selected_structures/al_runs`.

It has two batch steps:
1. run `pace_select` in each run folder (with automatic element detection and `m` capped by available snapshots)
2. unpack `selected.pkl.gz` and write `POSCAR` files for DFT (`POSCARS/dft_*/POSCAR`)

Run this notebook from the same folder as `01_select_structures.ipynb` and `02_make_al_runs.ipynb` (the folder that contains `selected_structures`).

In [1]:
from __future__ import annotations

import gzip
import pickle
import shlex
import subprocess
from pathlib import Path
from typing import Iterable

from ase import Atoms
from ase.io import read, write

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

In [2]:
CFG = {
    # Run this notebook from the folder that contains `selected_structures`
    'al_runs_dir': Path('selected_structures') / 'al_runs',
    'run_glob': 's*',

    # Pacemaker model files (relative to this notebook run directory, i.e. ./al)
    'model_yaml': Path('..') / 'output_potential.yaml',
    'model_asi': Path('..') / 'output_potential.asi',
    'pace_select_cmd': ['conda', 'run', '-n', 'ace_stable', 'pace_select'],

    # D-optimality selection settings
    'max_m': 25,                   # upper limit; actual m is capped by dump snapshots
    'require_dump': True,
    'skip_if_present': ['POSCARS', 'selected.pkl.gz'],  # for pace_select step only

    # Execution controls for pace_select
    'dry_run': True,               # set False to actually run pace_select
    'stop_on_error': False,
    'print_stdout_tail': 20,
    'print_stderr_tail': 40,

    # Export POSCARs from selected.pkl.gz (second step)
    'export_selected_poscars': True,
    'selected_pickle_name': 'selected.pkl.gz',
    'poscars_dirname': 'POSCARS',
    'dft_dir_prefix': 'dft_',
    'overwrite_poscars': False,
    'skip_export_if_poscars_exists': False,
}

CFG

{'al_runs_dir': PosixPath('selected_structures/al_runs'),
 'run_glob': 's*',
 'model_yaml': PosixPath('../output_potential.yaml'),
 'model_asi': PosixPath('../output_potential.asi'),
 'pace_select_cmd': ['conda', 'run', '-n', 'ace_stable', 'pace_select'],
 'max_m': 25,
 'require_dump': True,
 'skip_if_present': ['POSCARS', 'selected.pkl.gz'],
 'dry_run': True,
 'stop_on_error': False,
 'print_stdout_tail': 20,
 'print_stderr_tail': 40,
 'export_selected_poscars': True,
 'selected_pickle_name': 'selected.pkl.gz',
 'poscars_dirname': 'POSCARS',
 'dft_dir_prefix': 'dft_',
 'overwrite_poscars': False,
 'skip_export_if_poscars_exists': False}

In [3]:
def _unique_in_order(items: Iterable[str]) -> list[str]:
    out = []
    seen = set()
    for x in items:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def format_cmd(cmd: list[str]) -> str:
    return ' '.join(shlex.quote(x) for x in cmd)


def discover_run_dirs(cfg: dict) -> list[Path]:
    root = Path(cfg['al_runs_dir']).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f'al_runs directory not found: {root}')
    return [p for p in sorted(root.glob(cfg.get('run_glob', 's*'))) if p.is_dir()]


def infer_species(run_dir: Path) -> list[str]:
    poscar = run_dir / 'POSCAR'
    if poscar.is_file():
        atoms = read(poscar)
        return _unique_in_order(atoms.get_chemical_symbols())

    atoms_data = run_dir / 'atoms.data'
    if atoms_data.is_file():
        atoms = read(atoms_data, format='lammps-data')
        return _unique_in_order(atoms.get_chemical_symbols())

    raise FileNotFoundError(f'No POSCAR or atoms.data found in {run_dir}')


def count_dump_snapshots(dump_path: Path) -> int:
    n = 0
    with dump_path.open('r', errors='ignore') as fh:
        for line in fh:
            if line.startswith('ITEM: TIMESTEP'):
                n += 1
    return n


def build_pace_select_command(run_dir: Path, cfg: dict, species: list[str], m_eff: int) -> list[str]:
    model_yaml = Path(cfg['model_yaml']).resolve()
    model_asi = Path(cfg['model_asi']).resolve()
    if not model_yaml.is_file():
        raise FileNotFoundError(f'Model YAML not found: {model_yaml}')
    if not model_asi.is_file():
        raise FileNotFoundError(f'Model ASI not found: {model_asi}')

    return list(cfg['pace_select_cmd']) + [
        '-p', str(model_yaml),
        '-a', str(model_asi),
        '-e', ' '.join(species),
        '-m', str(int(m_eff)),
        'extrapolative_structures.dump',
    ]


def should_skip_pace_select(run_dir: Path, cfg: dict) -> tuple[bool, str]:
    for name in cfg.get('skip_if_present', []):
        if (run_dir / name).exists():
            return True, f'already has {name}'
    return False, ''

In [4]:
def run_pace_select_batch(cfg: dict):
    rows = []
    run_dirs = discover_run_dirs(cfg)
    print(f'Found {len(run_dirs)} candidate run folders in {Path(cfg["al_runs_dir"]).resolve()}')

    for run_dir in run_dirs:
        row = {
            'run_dir': str(run_dir),
            'status': 'pending',
            'snapshots': None,
            'm': None,
            'elements': None,
            'reason': None,
            'returncode': None,
            'command': None,
        }

        dump_path = run_dir / 'extrapolative_structures.dump'
        if cfg.get('require_dump', True) and not dump_path.is_file():
            row['status'] = 'skip'
            row['reason'] = 'missing extrapolative_structures.dump'
            rows.append(row)
            continue

        skip, reason = should_skip_pace_select(run_dir, cfg)
        if skip:
            row['status'] = 'skip'
            row['reason'] = reason
            rows.append(row)
            continue

        try:
            species = infer_species(run_dir)
            n_snap = count_dump_snapshots(dump_path)
        except Exception as exc:
            row['status'] = 'error'
            row['reason'] = f'{type(exc).__name__}: {exc}'
            rows.append(row)
            if cfg.get('stop_on_error'):
                raise
            continue

        row['elements'] = ' '.join(species)
        row['snapshots'] = int(n_snap)
        if n_snap <= 0:
            row['status'] = 'skip'
            row['reason'] = 'dump has 0 snapshots'
            rows.append(row)
            continue

        row['m'] = min(int(cfg['max_m']), int(n_snap))

        try:
            cmd = build_pace_select_command(run_dir, cfg, species, row['m'])
        except Exception as exc:
            row['status'] = 'error'
            row['reason'] = f'{type(exc).__name__}: {exc}'
            rows.append(row)
            if cfg.get('stop_on_error'):
                raise
            continue

        row['command'] = format_cmd(cmd)

        if cfg.get('dry_run', True):
            row['status'] = 'dry-run'
            rows.append(row)
            print(f"[dry-run] {run_dir.name}: m={row['m']}, elements={row['elements']}")
            print(f"         {row['command']}")
            continue

        print(f"[run] {run_dir.name}: m={row['m']}, elements={row['elements']}")
        proc = subprocess.run(cmd, cwd=run_dir, capture_output=True, text=True)
        row['returncode'] = int(proc.returncode)
        row['status'] = 'ok' if proc.returncode == 0 else 'error'
        if proc.returncode != 0:
            row['reason'] = 'pace_select returned non-zero'

        out_tail_n = int(cfg.get('print_stdout_tail', 0) or 0)
        err_tail_n = int(cfg.get('print_stderr_tail', 0) or 0)
        if proc.stdout and out_tail_n > 0:
            print('  stdout tail:')
            for line in proc.stdout.splitlines()[-out_tail_n:]:
                print('   ', line)
        if proc.stderr and err_tail_n > 0:
            print('  stderr tail:')
            for line in proc.stderr.splitlines()[-err_tail_n:]:
                print('   ', line)

        rows.append(row)
        if row['status'] == 'error' and cfg.get('stop_on_error'):
            break

    return rows

In [5]:
def _to_atoms(obj):
    # Matches the normalization pattern from inspect_selected.ipynb
    if isinstance(obj, Atoms):
        return obj.copy()
    try:
        return Atoms(obj)
    except Exception:
        return Atoms([obj])


def _format_dft_dir_name(idx, prefix='dft_') -> str:
    return f"{prefix}{idx}"


def export_poscars_from_selected_pickle(run_dir: Path, cfg: dict) -> dict:
    pkl_name = str(cfg.get('selected_pickle_name', 'selected.pkl.gz'))
    pkl_path = run_dir / pkl_name
    poscars_root = run_dir / str(cfg.get('poscars_dirname', 'POSCARS'))

    row = {
        'run_dir': str(run_dir),
        'status': 'pending',
        'selected_pickle': str(pkl_path),
        'poscars_dir': str(poscars_root),
        'n_selected': None,
        'n_written': 0,
        'atoms_col': None,
        'reason': None,
    }

    if not pkl_path.is_file():
        row['status'] = 'skip'
        row['reason'] = f'missing {pkl_name}'
        return row

    if poscars_root.exists() and cfg.get('skip_export_if_poscars_exists', False):
        row['status'] = 'skip'
        row['reason'] = f'{poscars_root.name} already exists'
        return row

    with gzip.open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    if not hasattr(data, 'columns'):
        raise TypeError(f'{pkl_path} does not contain a pandas DataFrame-like object')

    atoms_col = None
    for cand in ['ase_atoms', 'atoms', 'structure']:
        if cand in data.columns:
            atoms_col = cand
            break
    if atoms_col is None:
        raise KeyError(f'No atoms column found in {pkl_path}; tried ase_atoms/atoms/structure')

    row['atoms_col'] = atoms_col
    row['n_selected'] = int(len(data))
    poscars_root.mkdir(parents=True, exist_ok=True)

    overwrite = bool(cfg.get('overwrite_poscars', False))
    dft_prefix = str(cfg.get('dft_dir_prefix', 'dft_'))

    for idx, obj in data[atoms_col].items():
        atoms = _to_atoms(obj)
        dft_dir = poscars_root / _format_dft_dir_name(idx, prefix=dft_prefix)
        dft_dir.mkdir(parents=True, exist_ok=True)
        poscar_path = dft_dir / 'POSCAR'
        if poscar_path.exists() and not overwrite:
            continue
        write(poscar_path, atoms, format='vasp')
        row['n_written'] += 1

    row['status'] = 'ok'
    return row


def export_selected_poscars_batch(cfg: dict):
    rows = []
    run_dirs = discover_run_dirs(cfg)
    print(f'Export scan across {len(run_dirs)} run folders')

    for run_dir in run_dirs:
        try:
            row = export_poscars_from_selected_pickle(run_dir, cfg)
        except Exception as exc:
            row = {
                'run_dir': str(run_dir),
                'status': 'error',
                'selected_pickle': str(run_dir / str(cfg.get('selected_pickle_name', 'selected.pkl.gz'))),
                'poscars_dir': str(run_dir / str(cfg.get('poscars_dirname', 'POSCARS'))),
                'n_selected': None,
                'n_written': 0,
                'atoms_col': None,
                'reason': f'{type(exc).__name__}: {exc}',
            }
        rows.append(row)
        if row['status'] == 'ok':
            print(f"[export] {run_dir.name}: wrote {row['n_written']} POSCARs to {Path(row['poscars_dir']).name}")
        elif row['status'] == 'skip':
            print(f"[skip]   {run_dir.name}: {row['reason']}")
        else:
            print(f"[error]  {run_dir.name}: {row['reason']}")
            if cfg.get('stop_on_error'):
                break

    return rows

In [6]:
pace_rows = run_pace_select_batch(CFG)

if pd is not None:
    pace_df = pd.DataFrame(pace_rows)
    display(pace_df)
    display(pace_df['status'].value_counts(dropna=False).rename('count'))
else:
    from pprint import pprint
    pprint(pace_rows)

Found 1 candidate run folders in /nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs
[dry-run] s003_binary_phases_LiO8_s1030_k001: m=9, elements=Li O
         conda run -n ace_stable pace_select -p /nfshome/sicolo/work/MLIP/train/output_potential.yaml -a /nfshome/sicolo/work/MLIP/train/output_potential.asi -e 'Li O' -m 9 extrapolative_structures.dump


,run_dir,status,snapshots,m,elements,reason,returncode,command
0,/nfshome/sicolo/work/MLIP/train/al_workflow/se...,dry-run,9,9,Li O,None,None,conda run -n ace_stable pace_select -p /nfshom...


status
dry-run    1
Name: count, dtype: int64

In [10]:
# After verifying the dry run above:
CFG['dry_run'] = False
pace_rows = run_pace_select_batch(CFG)

Found 1 candidate run folders in /nfshome/sicolo/work/MLIP/train/al_workflow/selected_structures/al_runs
[run] s003_binary_phases_LiO8_s1030_k001: m=9, elements=Li O
  stdout tail:
    Species type: 0
    Iter 0/300: abs_max = 2.893 (tol = 1.010)
    Iter 10/300: abs_max = 1.139 (tol = 1.010)
    Species type: 1
    Species type: 2
    Iter 0/300: abs_max = 2.806 (tol = 1.010)
    Iter 10/300: abs_max = 1.423 (tol = 1.010)
    Iter 20/300: abs_max = 1.160 (tol = 1.010)
    Iter 30/300: abs_max = 1.176 (tol = 1.010)
    Iter 40/300: abs_max = 1.092 (tol = 1.010)
    Iter 50/300: abs_max = 1.048 (tol = 1.010)
    
  stderr tail:
    /nfshome/sicolo/miniconda3/envs/ace_stable/lib/python3.9/site-packages/pyace/multispecies_basisextension.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
      im

In [11]:
export_rows = []
if CFG.get('export_selected_poscars', True):
    export_rows = export_selected_poscars_batch(CFG)

    if pd is not None:
        export_df = pd.DataFrame(export_rows)
        display(export_df)
        if len(export_df):
            display(export_df['status'].value_counts(dropna=False).rename('count'))
    else:
        from pprint import pprint
        pprint(export_rows)
else:
    print('POSCAR export step disabled (CFG["export_selected_poscars"] = False)')

Export scan across 1 run folders
[export] s003_binary_phases_LiO8_s1030_k001: wrote 9 POSCARs to POSCARS


/tmp/ipykernel_97404/516717929.py:42: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


,run_dir,status,selected_pickle,poscars_dir,n_selected,n_written,atoms_col,reason
0,/nfshome/sicolo/work/MLIP/train/al_workflow/se...,ok,/nfshome/sicolo/work/MLIP/train/al_workflow/se...,/nfshome/sicolo/work/MLIP/train/al_workflow/se...,9,9,ase_atoms,None


status
ok    1
Name: count, dtype: int64

In [9]:
# Optional: re-run only the export step after pace_select finishes
# CFG['export_selected_poscars'] = True
# export_rows = export_selected_poscars_batch(CFG)